In [ ]:
# ============================================================
# Add RDF-star circa annotations to selected properties
# ============================================================

from pathlib import Path
import sys
import subprocess

try:
    from rdflib import Graph, Namespace, URIRef, Literal
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "rdflib"])
    from rdflib import Graph, Namespace, URIRef, Literal


# ============================================================
# CONFIGURATION
# ============================================================

INPUT_TTL = Path("wendangData.ttl")
OUTPUT_TTL = Path("wendangDataset.ttl")
REPORT_TXT = Path("accuracy-annotation-report.txt")


# ============================================================
# NAMESPACES IN THE DATASET
# ============================================================

NS1 = Namespace("https://w3id.org/write/ontology#")
NS2 = Namespace("http://www.wikidata.org/prop/direct/")
NS3_HTTP = Namespace("http://www.wikidata.org/entity/")
NS3_HTTPS = Namespace("https://www.wikidata.org/entity/")
XSD = Namespace("http://www.w3.org/2001/XMLSchema#")

P1480 = NS2["P1480"]

P571 = NS2["P571"]
NUMBER_OF_CHARACTERS = NS1["numberOfCharacters"]
ACCURACY = NS1["accuracy"]

ACCURACY_MARKERS = {
    NS3_HTTP["Q5727902"]: "circa",
    NS3_HTTPS["Q5727902"]: "circa",
    NS3_HTTP["Q18122778"]: "presumably",
    NS3_HTTPS["Q18122778"]: "presumably",
}

PROPERTIES_TO_ANNOTATE = [
    P571,
    NUMBER_OF_CHARACTERS,
]

PREFIXES = {
    "geosparql": "http://www.opengis.net/ont/geosparql#",
    "ns1": "https://w3id.org/write/ontology#",
    "ns2": "http://www.wikidata.org/prop/direct/",
    "ns3": "http://www.wikidata.org/entity/",
    "xsd": "http://www.w3.org/2001/XMLSchema#",
    "rdfs": "http://www.w3.org/2000/01/rdf-schema#",
}


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def turtle_term(term):
    if isinstance(term, URIRef):
        iri = str(term)

        for prefix, namespace in PREFIXES.items():
            if iri.startswith(namespace):
                local = iri[len(namespace):]
                return f"{prefix}:{local}"

        return f"<{iri}>"

    if isinstance(term, Literal):
        lexical = str(term)
        lexical = lexical.replace("\\", "\\\\").replace('"', '\\"')

        if term.language:
            return f'"{lexical}"@{term.language}'

        if term.datatype:
            return f'"{lexical}"^^{turtle_term(term.datatype)}'

        return f'"{lexical}"'

    raise TypeError(f"Unsupported RDF term: {term!r}")


def ensure_prefixes(original_text):
    prefix_lines = {
        "geosparql:": "@prefix geosparql: <http://www.opengis.net/ont/geosparql#> .",
        "ns1:": "@prefix ns1: <https://w3id.org/write/ontology#> .",
        "ns2:": "@prefix ns2: <http://www.wikidata.org/prop/direct/> .",
        "ns3:": "@prefix ns3: <http://www.wikidata.org/entity/> .",
        "xsd:": "@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .",
        "rdfs:": "@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .",
    }

    missing = []

    for prefix_name, prefix_line in prefix_lines.items():
        if f"@prefix {prefix_name}" not in original_text and f"PREFIX {prefix_name}" not in original_text:
            missing.append(prefix_line)

    if missing:
        return "\n".join(missing) + "\n\n" + original_text

    return original_text


# ============================================================
# LOAD DATA
# ============================================================

if not INPUT_TTL.exists():
    raise FileNotFoundError(
        f"Input file not found: {INPUT_TTL.resolve()}\n"
        "Change INPUT_TTL to your actual TTL filename."
    )

g = Graph()
g.parse(INPUT_TTL, format="turtle")

print(f"Loaded {len(g):,} triples from {INPUT_TTL}")


# ============================================================
# FIND P1480 MARKERS AND CREATE STATEMENT ANNOTATIONS
# ============================================================

annotations = []

for item, marker in g.subject_objects(P1480):
    if marker not in ACCURACY_MARKERS:
        continue

    accuracy_label = ACCURACY_MARKERS[marker]

    for prop in PROPERTIES_TO_ANNOTATE:
        for value in g.objects(item, prop):
            annotations.append((item, prop, value, accuracy_label, marker))

seen = set()
unique_annotations = []

for s, p, o, label, marker in annotations:
    key = (s, p, o, label)
    if key not in seen:
        seen.add(key)
        unique_annotations.append((s, p, o, label, marker))

print(f"Generated {len(unique_annotations):,} RDF-star statement annotations")


# ============================================================
# WRITE RDF-STAR ANNOTATION BLOCK
# ============================================================

annotation_lines = [
    "",
    "",
    "### RDF-star accuracy annotations generated automatically",
    "### These annotations attach ns1:accuracy to the exact statement.",
    '### Pattern: << item property value >> ns1:accuracy "circa" .',
    "",
]

for s, p, o, label, marker in unique_annotations:
    annotation_lines.append(
        f"<< {turtle_term(s)} {turtle_term(p)} {turtle_term(o)} >> "
        f'ns1:accuracy "{label}" .'
    )

annotation_block = "\n".join(annotation_lines)


# ============================================================
# WRITE OUTPUT TTL
# ============================================================

original_ttl = INPUT_TTL.read_text(encoding="utf-8")
final_ttl = ensure_prefixes(original_ttl).rstrip() + "\n" + annotation_block + "\n"

OUTPUT_TTL.write_text(final_ttl, encoding="utf-8")

print(f"Wrote new Sampo TTL file: {OUTPUT_TTL}")
print(f"Added {len(unique_annotations):,} RDF-star annotations")


# ============================================================
# WRITE REPORT
# ============================================================

report_lines = [
    "Accuracy annotation report",
    "==========================",
    "",
    f"Input TTL: {INPUT_TTL.resolve()}",
    f"Output TTL: {OUTPUT_TTL.resolve()}",
    f"Loaded triples: {len(g):,}",
    f"Generated RDF-star annotations: {len(unique_annotations):,}",
    "",
    "Generated annotations:",
    "",
]

for s, p, o, label, marker in unique_annotations:
    report_lines.append(
        f"<< {turtle_term(s)} {turtle_term(p)} {turtle_term(o)} >> "
        f'ns1:accuracy "{label}" .'
        f"  # from marker {marker}"
    )

REPORT_TXT.write_text("\n".join(report_lines), encoding="utf-8")

print(f"Wrote report file: {REPORT_TXT}")


# ============================================================
# PREVIEW
# ============================================================

print("\nPreview of generated annotations:")
print("---------------------------------")

if unique_annotations:
    for s, p, o, label, marker in unique_annotations[:30]:
        print(
            f"<< {turtle_term(s)} {turtle_term(p)} {turtle_term(o)} >> "
            f'ns1:accuracy "{label}" .'
        )
else:
    print("No annotations generated.")
    print("The script found no items with P1480 values matching:")
    print("- Q5727902 / circa")
    print("- Q18122778 / presumably")
    print("or those items did not have ns2:P571 or ns1:numberOfCharacters.")